# rdepth — 순환 깊이 30M 프로토타입 (Colab) v2

**사용법**: ① 런타임 → 런타임 유형 변경 → **T4 GPU** ② **런타임 → 모두 실행** ③ 초반에 드라이브 허용 팝업과 `rdepth_code.zip` 업로드 창만 처리

- 체크포인트·로그·데이터가 전부 Drive(`MyDrive/rdepth_out`)에 저장됩니다.
- **세션이 끊기거나 오류로 멈추면: 그냥 다시 '모두 실행'** — 받은 데이터는 재사용, 학습은 체크포인트에서 이어짐.
- 학습 셀 3개(small/loop/large)는 진행 로그가 실시간으로 출력됩니다.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs(DRIVE, exist_ok=True)
zpath = f'{DRIVE}/rdepth_code.zip'
if not os.path.exists(zpath):
    from google.colab import files
    print('rdepth_code.zip 파일을 선택해 주세요:')
    up = files.upload()
    shutil.move(list(up)[0], zpath)
!unzip -q -o {zpath} -d /content/rdepth
%cd /content/rdepth
print('코드 준비 완료')

In [ ]:
%cd /content/rdepth
!pip -q install tokenizers pytest 2>/dev/null
!python -m pytest tests/test_model.py -q

In [ ]:
%cd /content/rdepth
import os
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs('data', exist_ok=True)
if os.path.exists(f'{DRIVE}/train.bin'):
    print('Drive 캐시에서 데이터 복사 (~1분)...')
    !cp {DRIVE}/tok4096.json {DRIVE}/val.bin {DRIVE}/train.bin data/
else:
    print('TinyStories 다운로드+인코딩 (~15분, 로그가 아래 출력됨)...')
    !python prepare_data.py
    !cp data/tok4096.json data/val.bin data/train.bin {DRIVE}/
!python -m pytest tests/test_data.py -q

In [ ]:
import torch, os
os.environ['RDEPTH_OUT'] = '/content/drive/MyDrive/rdepth_out'
# T4(Turing)는 is_bf16_supported()가 True여도 에뮬레이션이라 느림 — Ampere(A100 등, sm80+)만 bf16
cap = torch.cuda.get_device_capability()
DTYPE = 'bf16' if cap[0] >= 8 else 'fp16'
print('GPU =', torch.cuda.get_device_name(0), cap, '| dtype =', DTYPE, '| 출력 위치 =', os.environ['RDEPTH_OUT'])

In [ ]:
%cd /content/rdepth
!python train.py --run small --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

In [ ]:
%cd /content/rdepth
!python train.py --run loop --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

In [ ]:
%cd /content/rdepth
!python train.py --run large --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

In [ ]:
import csv, os
out = '/content/drive/MyDrive/rdepth_out'
rows = {}
for r in ['small', 'loop', 'large']:
    p = f'{out}/logs/{r}.csv'
    if os.path.exists(p):
        with open(p) as f: data = list(csv.DictReader(f))
        if data: rows[r] = min(float(d['val_loss']) for d in data)
print('최저 val loss:', rows)
if len(rows) == 3:
    s, l, g = rows['small'], rows['loop'], rows['large']
    rec = (s - l) / (s - g) * 100 if s != g else float('nan')
    print(f"kill-gate(loop<small): {'PASS' if l < s else 'FAIL'}   회복률 {rec:.1f}%")